# Разбор и подготовка Retail.xlsx для BI-отчёта

Перед нами выгрузка транзакций онлайн-ритейлера за два года: почти миллион строк, десять полей, ноль документации. Прежде чем загружать это в Yandex DataLens, нужно разобраться, что внутри: где настоящие продажи, где возвраты, где бухгалтерский мусор, а где — целый сегмент покупателей, о которых источник ничего не знает.

Этот notebook — рабочий журнал такого разбора. Каждый раздел — отдельная проверка с выводом и решением. Финальная логика трансформации живёт в модулях `src/`, а здесь фиксируется ход рассуждений: что нашли, почему это важно и что с этим сделали.

## 1. Загрузка и подготовка

Читаем исходный Excel, применяем базовую нормализацию текста (trim, регистр, пробелы) и размечаем каждую строку по типу операции. Всё это — вызовы модулей из `src/`, чтобы notebook оставался читаемым.

In [13]:
# File: c:\Git\MFTI\DataVizualize\Финальный проект_ИАВД-11\Финальный проект_источники\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
# ruff: noqa: E402
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    for candidate in [project_root, *project_root.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            project_root = candidate
            break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.classification import classify_line_type
from src.config import PipelineConfig
from src.io_utils import load_retail_data
from src.normalization import apply_normalization
from src.pipeline import run_pipeline
from src.profiling import (
    build_basic_profile,
    build_country_mapping_table,
    build_extreme_rows,
    build_last_month_summary,
    build_line_type_summary,
    build_missingness_profile,
    build_stock_description_issues,
    build_text_noise_summary,
    find_anonymous_transactions,
    find_bad_debt_candidates,
    find_business_duplicates,
    find_exact_duplicates,
    find_missing_description_rows,
    find_return_candidates,
    find_service_code_rows,
    find_zero_price_rows,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)

cfg = PipelineConfig()
raw_df, source_path = load_retail_data(cfg)
normalized_df = apply_normalization(raw_df, cfg)
audited_df = classify_line_type(normalized_df, cfg)
source_path

WindowsPath('C:/Git/MFTI/DataVizualize/final_project/retail_bi_pipeline/data/raw/Retail.xlsx')

## 2. Первый взгляд на данные

Начнём с самого простого: сколько строк, какие поля, за какой период, где пропуски. На этом этапе не чистим — только смотрим и запоминаем, что выглядит подозрительно.

In [14]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
print("source:", source_path)
print("shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes.to_frame("dtype"))
display(build_basic_profile(raw_df))
display(build_missingness_profile(raw_df))

source: C:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\data\raw\Retail.xlsx
shape: (947217, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085.0,United Kingdom,organic,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,organic,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,organic,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,organic,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,organic,False


,dtype
Invoice,string[python]
StockCode,string[python]
Description,string[python]
Quantity,Int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,string[python]
Channel,string[python]
rnd,boolean


,metric,value
0,row_count,947217
1,column_count,10
2,invoice_unique,48753
3,customer_unique_non_null,5695
4,country_unique,43
5,date_min,2009-12-01 00:00:00
6,date_max,2011-10-27 00:00:00
7,invoice_time_unique,1


,column_name,missing_count,missing_pct
0,Channel,214018,0.225944
1,Customer ID,214018,0.225944
2,Description,4287,0.004526
3,Country,0,0.000000
4,Invoice,0,0.000000
5,InvoiceDate,0,0.000000
6,Price,0,0.000000
7,Quantity,0,0.000000
8,StockCode,0,0.000000
9,rnd,0,0.000000


### Что видим

Набор большой — **947 тысяч строк** за период с декабря 2009 по октябрь 2011. Десять полей, из которых ключевые (`Invoice`, `StockCode`, `Quantity`, `Price`, `InvoiceDate`) заполнены полностью.

Но сразу бросается в глаза аномалия: **214 018 строк (22.6%) не имеют ни `Customer ID`, ни `Channel`**. Причём пропуски идут строго парой — нет ни одной строки, где клиент есть, а канал пропущен, или наоборот. Это не случайные дыры в данных, а целый пласт транзакций, которые источник передал без клиентского контура. Почти четверть набора.

Ещё `4 287` строк без `Description` — немного на фоне миллиона, но для товарной аналитики это значимо: транзакция есть, а что именно продали — неизвестно.

Вывод: данные не «грязные» в привычном смысле. Они скорее *неоднородные* — в одной таблице смешаны полностью атрибутированные продажи и операции, о которых известно гораздо меньше.

### Загадочное поле `rnd`

В профиле данных есть колонка `rnd` — булев тип, без пропусков, документации нет. Посмотрим, что в ней.

In [ ]:
print("rnd dtype:", raw_df["rnd"].dtype)
print("rnd value_counts:")
display(raw_df["rnd"].value_counts())
print(f"\nrnd=True: {raw_df['rnd'].sum()} строк ({raw_df['rnd'].mean():.1%})")

# Распределение по месяцам — равномерное?
rnd_by_month = (
    raw_df.assign(ym=raw_df["InvoiceDate"].dt.to_period("M").astype(str))
    .groupby("ym")["rnd"]
    .mean()
    .mul(100)
    .round(1)
    .rename("pct_rnd_true")
)
display(rnd_by_month)

Около **37 тысяч строк (3.9%)** помечены `rnd=True`. Доля стабильна по месяцам (3.6–5.1%) — распределение равномерное, не привязано к сезону или конкретному событию. Название `rnd` наводит на мысль о «random», но это пока гипотеза. Запомним и будем отслеживать, как этот флаг соотносится с аномалиями дальше.

## 3. Дубликаты: технический шум или реальные операции?

В датасете на миллион строк дубликаты — не редкость. Вопрос в том, *какие именно* повторы перед нами: сбой при выгрузке (вся строка скопирована дословно) или два реальных заказа, которые просто совпали по всем параметрам.

Поэтому проверяем два уровня:
- **Полный дубликат** — буквальное совпадение всех 10 полей, включая технический флаг `rnd`. Это почти наверняка копия.
- **Дубликат по бизнес-ключу** — совпадение по `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`, `Channel`. Здесь уже возможны и настоящие повторные покупки.

In [15]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
exact_dups = find_exact_duplicates(raw_df)
business_dups = find_business_duplicates(raw_df, cfg)
print("exact duplicates:", len(exact_dups))
print("business duplicates:", len(business_dups))
display(exact_dups.head())
display(business_dups.head())

exact duplicates: 59730
business duplicates: 63858


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,sales,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,sales,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,sales,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,sales,False


In [ ]:
# Проверяем: как rnd соотносится с дубликатами?
_is_dup = raw_df.duplicated(keep=False)
print("=== rnd vs exact duplicate ===")
display(pd.crosstab(raw_df["rnd"], _is_dup.rename("is_duplicate"), margins=True))
print(f"\nrnd=True среди дубликатов: {raw_df.loc[_is_dup, 'rnd'].mean():.1%}")
print(f"rnd=True в среднем по набору: {raw_df['rnd'].mean():.1%}")

### Что видим

Масштаб серьёзный: **59 730 полных дублей** — это 6.3% всего набора. Не единичные сбои, а системная проблема выгрузки. По бизнес-ключу дублей чуть больше — **63 858**, разница в ~4 000 строк приходится на записи, которые совпадают по операции, но различаются значением `rnd`.

Если оставить полные дубли в факте продаж, выручка и количество проданных единиц будут завышены. Это не мелочь — при средней строке ~19 рублей выручки набегает ощутимая сумма.

**Наблюдение по `rnd`:** доля `rnd=True` среди дубликатов заметно ниже средней по набору. Пока не делаем выводов — просто фиксируем.

**Решение:** полные дубли снимаем перед построением фактов. Бизнес-дубли не удаляем, но помечаем флагом `is_business_duplicate` — пусть аналитик в BI сам решит, фильтровать их или нет.

## 4. Возвраты: два признака, которые не совпадают

Интуитивный способ найти возвраты — искать инвойсы с префиксом `C` (cancel). Но есть и второй сигнал: отрицательное `Quantity`. Если оба признака дают одну и ту же выборку — отлично, берём любой. Если нет — нужно разбираться.

In [16]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
returns_df = find_return_candidates(raw_df)
invoice_c = raw_df["Invoice"].astype("string").fillna("").str.upper().str.startswith("C")
qty_negative = raw_df["Quantity"] < 0
print("invoice startswith C:", int(invoice_c.sum()))
print("quantity < 0:", int(qty_negative.sum()))
print("negative qty but not C:", int((qty_negative & ~invoice_c).sum()))
print("C invoice but positive qty:", int((invoice_c & (raw_df["Quantity"] > 0)).sum()))
display(returns_df.head())

invoice startswith C: 17980
quantity < 0: 21234
negative qty but not C: 3255
C invoice but positive qty: 1


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01,2.95,16321.0,Australia,sales,False
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01,1.65,16321.0,Australia,sales,False
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01,4.25,16321.0,Australia,sales,False
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01,2.10,16321.0,Australia,sales,False
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01,2.95,16321.0,Australia,sales,False


### Что видим

Признаки **не совпадают**. Инвойсов на `C` — 17 980, строк с отрицательным количеством — 21 234. Разница: **3 255 строк с минусом, но без префикса `C`** — это складские списания, коррекции (`damaged`, `lost`, `thrown away`) и прочие операции, которые уменьшают остаток, но формально не являются клиентским возвратом. Плюс одна аномальная строка с `C`-инвойсом и положительным количеством.

**Решение:** для классификации возвратов используем комбинированное правило — `Quantity < 0` **ИЛИ** `Invoice` начинается с `C`. Так не потеряем ни клиентские возвраты, ни складские корректировки.

## 5. Скрытый бухгалтерский слой

Помимо товарных операций и возвратов, в источнике могут прятаться записи совсем другой природы — бухгалтерские корректировки. Их легко пропустить, потому что они лежат в той же таблице и отличаются только нетипичными значениями `Invoice`, `StockCode` и `Price`.

In [17]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
bad_debt_df = find_bad_debt_candidates(raw_df)
display(bad_debt_df)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
179403,A506401,B,Adjust bad debt,1,2010-04-29,-53594.36,NaN,United Kingdom,<NA>,True
276274,A516228,B,Adjust bad debt,1,2010-07-19,-44031.79,NaN,United Kingdom,<NA>,True
403472,A528059,B,Adjust bad debt,1,2010-10-20,-38925.87,NaN,United Kingdom,<NA>,True
825443,A563185,B,Adjust bad debt,1,2011-08-12,11062.06,NaN,United Kingdom,<NA>,False
825444,A563186,B,Adjust bad debt,1,2011-08-12,-11062.06,NaN,United Kingdom,<NA>,True
825445,A563187,B,Adjust bad debt,1,2011-08-12,-11062.06,NaN,United Kingdom,<NA>,True


In [ ]:
# rnd vs отрицательная цена
neg_price = raw_df["Price"] < 0
print("=== rnd vs Price < 0 ===")
display(pd.crosstab(raw_df["rnd"], neg_price.rename("negative_price"), margins=True))
print(f"\nrnd=True среди Price<0: {raw_df.loc[neg_price, 'rnd'].mean():.1%}")

### Что видим

Шесть строк с `Invoice` на `A`, `StockCode = B`, описанием `Adjust bad debt` и суммами до **−53 594**. Это списания безнадёжной дебиторской задолженности — чисто бухгалтерская проводка, которая не имеет отношения ни к продаже товара, ни к возврату. Суммарно по ним проходит около **−148 тысяч** — если не отделить их от товарного контура, они исказят агрегаты выручки.

**Наблюдение по `rnd`:** 5 из 6 строк bad debt имеют `rnd=True`. Все строки с отрицательной ценой — тоже `rnd=True`. Совпадение с `rnd` становится всё заметнее.

**Решение:** выносим в отдельный тип `bad_debt` и не смешиваем ни с продажами, ни с возвратами.

## 6. Нулевые цены и служебные коды: что ещё не является продажей

В разделах 4–5 мы отделили возвраты и бухгалтерию. Но в таблице есть и третий тип «не-продаж»: строки с `Price = 0` и записи с явно нетоварными `StockCode` вроде `POST`, `DOT`, `D`, `M`. Проверим их масштаб.

In [18]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
zero_price_df = find_zero_price_rows(raw_df)
service_df = find_service_code_rows(raw_df, cfg)
print("zero price rows:", len(zero_price_df))
print("service code rows:", len(service_df))
display(zero_price_df.head())
display(service_df[["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(20))

zero price rows: 5837
service code rows: 5173


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
263,489464,21733,85123a mixed,-96,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
283,489463,71477,short,-240,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
284,489467,85123A,21733 mixed,-192,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
470,489521,21646,<NA>,-50,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3114,489655,20683,<NA>,-44,2009-12-01,0.0,NaN,United Kingdom,<NA>,True


,Invoice,StockCode,Description,Quantity,Price
89,489439,POST,POSTAGE,3,18.00
126,489444,POST,POSTAGE,1,141.00
173,489447,POST,POSTAGE,1,130.00
625,489526,POST,POSTAGE,6,18.00
735,C489535,D,Discount,-1,9.00
736,C489535,D,Discount,-1,19.00
927,C489538,POST,POSTAGE,-1,9.58
1244,489557,POST,POSTAGE,4,18.00
2379,489597,DOT,DOTCOM POSTAGE,1,647.19
2539,489600,DOT,DOTCOM POSTAGE,1,55.96


In [ ]:
# rnd vs нулевая цена
zero_price = raw_df["Price"] == 0
print("=== rnd vs Price == 0 ===")
display(pd.crosstab(raw_df["rnd"], zero_price.rename("zero_price"), margins=True))
print(f"\nrnd=True среди Price=0: {raw_df.loc[zero_price, 'rnd'].mean():.1%}")

### Что видим

**5 837 строк** с нулевой ценой и **5 173 строки** со служебными кодами — это не редкие исключения, а полноценные потоки данных. Среди них:

- **Доставка** (`POST`, `DOT`) — почтовые расходы и Dotcom-логистика. У них есть цена, и для финансового анализа они важны, но к товарной выручке не относятся.
- **Скидки** (`D`) — отрицательные суммы, привязанные к конкретным инвойсам.
- **Ручные корректировки** (`M`, `ADJUST`, `S`) — правки оператора, компенсации, образцы.
- **Комиссии** (`AMAZONFEE`, `BANK CHARGES`) — удержания маркетплейсов и банков.
- **Строки с нулевой ценой** — часто совпадают с пустым `Description` и пустым `Customer ID`; похожи на внутренние складские движения.

**Наблюдение по `rnd`:** *все* 5 837 строк с нулевой ценой имеют `rnd=True`. Это 100% — не корреляция, а полное совпадение. В `rnd=False` нет ни одной строки с `Price = 0`.

**Решение:** все эти строки получают свой `line_type` (shipping, discount, manual_adjustment и т.д.) и уходят в отдельную факт-таблицу `fact_service_lines`. Товарные KPI считаем только по `fact_sales_lines`.

## 7. 214 тысяч анонимных транзакций

Вернёмся к главной аномалии из раздела 2: пятая часть всех строк не имеет `Customer ID`. Это не баг одной колонки — вместе с клиентом пропадает и `Channel`. Нужно понять, что это за операции и можно ли с ними работать.

In [19]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
anonymous_df = find_anonymous_transactions(raw_df)
missing_desc_df = find_missing_description_rows(raw_df)
print("anonymous rows:", len(anonymous_df))
print("missing description rows:", len(missing_desc_df))
print(
    "channel missing & customer missing:",
    int((raw_df["Channel"].isna() & raw_df["Customer ID"].isna()).sum()),
)
print(
    "channel missing & customer present:",
    int((raw_df["Channel"].isna() & raw_df["Customer ID"].notna()).sum()),
)
display(anonymous_df.head())
display(missing_desc_df.head())

anonymous rows: 214018
missing description rows: 4287
channel missing & customer missing: 214018
channel missing & customer present: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
263,489464,21733,85123a mixed,-96,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
283,489463,71477,short,-240,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
284,489467,85123A,21733 mixed,-192,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
470,489521,21646,<NA>,-50,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01,0.55,NaN,United Kingdom,<NA>,False


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
470,489521,21646,<NA>,-50,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3114,489655,20683,<NA>,-44,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3161,489659,21350,<NA>,230,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3731,489781,84292,<NA>,17,2009-12-02,0.0,NaN,United Kingdom,<NA>,True
4296,489806,18010,<NA>,-770,2009-12-02,0.0,NaN,United Kingdom,<NA>,True


In [ ]:
# rnd vs пропуски Description и Customer ID
desc_na = raw_df["Description"].isna()
cust_na = raw_df["Customer ID"].isna()

print("=== rnd vs Description is NA ===")
display(pd.crosstab(raw_df["rnd"], desc_na.rename("desc_missing"), margins=True))
print(f"rnd=True среди Description=NA: {raw_df.loc[desc_na, 'rnd'].mean():.1%}")

print("\n=== rnd vs Customer ID is NA ===")
display(pd.crosstab(raw_df["rnd"], cust_na.rename("customer_missing"), margins=True))
print(f"rnd=True среди анонимных: {raw_df.loc[cust_na, 'rnd'].mean():.1%}")

### Что видим

Подтверждаем: `Customer ID` и `Channel` пропущены **ровно в одних и тех же** 214 018 строках. Ни одного случая, когда есть клиент без канала или канал без клиента. Это системное поведение источника: часть транзакций проходит анонимно — вероятно, розничные покупки без регистрации или оптовые операции без клиентского учёта.

Выбрасывать их нельзя — это 22.6% данных, а значит, существенная доля выручки. Но и притворяться, что у них есть клиент, тоже не стоит.

Отдельно — **4 287 строк без `Description`**. Все они анонимные, почти все с `Price = 0`. Это внутренние складские операции, в которых товарный атрибут не заполнен.

**Наблюдение по `rnd`:** 100% строк без `Description` — это `rnd=True`. Среди анонимных транзакций доля `rnd=True` выше средней, но не 100% — анонимность и `rnd` пересекаются, но не тождественны. Анонимные клиенты — реальное свойство данных; пустые описания — свойство `rnd`-слоя.

**Решения:**
- `Customer ID = NA` заменяем на явную метку `ANONYMOUS` — это не выдуманный клиент, а маркер анонимного сегмента.
- `Channel = NA` заменяем на `unknown` — не угадываем, а фиксируем факт.
- `Description = NA` оставляем пустым, но помечаем флагом `is_description_placeholder`. Туда же попадают служебные подписи вроде `DAMAGED`, `FOUND`, `CHECK` — они не являются названиями товаров.

## 8. Товарный справочник: почему `Description` ненадёжен

Для BI нужен справочник товаров. Логично строить его по `StockCode` + `Description`. Но прежде чем доверять этим полям, проверим, насколько они согласованы: один код — одно имя, и наоборот.

In [20]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_stock_description_issues(audited_df).head(20))
display(build_text_noise_summary(raw_df))
display(build_country_mapping_table(audited_df))

,issue_type,entity,variant_count
1132,description_to_many_stock_codes,<NA>,2417
1041,description_to_many_stock_codes,?,88
1064,description_to_many_stock_codes,DAMAGED,85
1065,description_to_many_stock_codes,DAMAGES,70
1091,description_to_many_stock_codes,MISSING,28
1077,description_to_many_stock_codes,FOUND,27
1055,description_to_many_stock_codes,CHECK,22
1116,description_to_many_stock_codes,SOLD AS SET ON DOTCOM,20
1045,description_to_many_stock_codes,ADJUSTMENT,12
1071,description_to_many_stock_codes,DOTCOM,11


,metric,value
0,description_trim_changed,184479
1,description_multiple_spaces,48058
2,stock_code_not_upper,3124


,country_raw,country_norm
0,Australia,Australia
1,Austria,Austria
2,Bahrain,Bahrain
3,Belgium,Belgium
4,Bermuda,Bermuda
...,...,...
38,United Arab Emirates,United Arab Emirates
39,United Kingdom,United Kingdom
40,USA,United States
41,Unspecified,Unknown


### Что видим

Справочник грязный в двух направлениях:

**Текстовый шум.** 184 тысячи описаний меняются просто от удаления пробелов по краям, ещё 48 тысяч содержат двойные пробелы внутри, 3 124 кода записаны в разном регистре (`47566B` vs `47566b`). Без нормализации группировки в BI будут дробиться на фантомные варианты одного и того же товара.

**Структурная проблема.** Одно `Description` может относиться к десяткам `StockCode` — но в топе таких «описаний» не товарные имена, а служебные пометки: `DAMAGED` (85 кодов), `MISSING` (28), `FOUND` (27), `CHECK` (22). Вместо того чтобы описывать товар, поле `Description` использовалось как комментарий оператора.

Обратная ситуация тоже есть: один `StockCode` может иметь и нормальное имя (`REGENCY CAKESTAND 3 TIER`), и мусорные подписи (`damaged`, `faulty`, `smashed`).

**Решение:** справочник товаров строим вокруг нормализованного `StockCode`. Для каждого кода выбираем каноническое имя — самое частое валидное описание среди продаж и возвратов. `Description` остаётся вспомогательным полем, но не ключом.

Страны нормализуем по словарю: `EIRE → Ireland`, `USA → United States`, `RSA → South Africa`, `Unspecified → Unknown`.

## 9. Экстремумы: 74 тысячи единиц в одном заказе

В любом крупном датасете найдутся строки с подозрительно большими числами. Вопрос — это ошибки ввода или реальные операции? Проверяем топ по `Quantity` и `Price`, а заодно — полноту последнего месяца.

In [21]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_extreme_rows(raw_df, top_n=10))
display(build_last_month_summary(raw_df))

,extreme_metric,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country,Channel
0,quantity,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom,organic
1,quantity,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346.0,United Kingdom,organic
2,quantity,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0,Denmark,sales
3,quantity,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0,Denmark,sales
4,quantity,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0,Denmark,sales
5,quantity,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0,Denmark,sales
6,quantity,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0,Denmark,sales
7,quantity,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,NaN,United Kingdom,<NA>
8,quantity,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0,United Kingdom,performance
9,quantity,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,17940.0,United Kingdom,performance


,max_invoice_date,last_month_start,calendar_month_end,is_last_month_complete
0,2011-10-27,2011-10-01,2011-10-31,False


### Что видим

Рекордсмен по количеству — **74 215 единиц** `MEDIUM CERAMIC TOP STORAGE JAR` (инвойс 541431, клиент 12346). Но рядом лежит зеркальная отмена на −74 215 единиц (инвойс C541433). Это полное сторно крупной оптовой операции, а не ошибка набора.

Следующие по величине — заказы на 12–19 тысяч единиц от одного датского клиента (13902) по 0.10 за штуку. Похоже на промо-партию или оптовый тестовый заказ.

По цене топ занимают уже знакомые строки: bad debt (до −53 594), ручные корректировки (`M`) на десятки тысяч и комиссии Amazon/банков. Все они уже классифицированы как сервисные в разделах 5–6.

**Последний месяц (октябрь 2011) неполный** — данные обрываются 27-го числа. Если сравнивать его с полными месяцами на графике, он будет выглядеть как провал. Помечаем флагом `is_last_incomplete_month`.

**Решение:** экстремумы не обрезаем — они объяснимы. Но сохраняем их в QA-таблице для контроля.

## 10. Итоговая раскладка: куда попала каждая строка

После всех проверок — дубликатов, возвратов, бухгалтерии, служебных кодов, анонимных транзакций — смотрим на финальную классификацию `line_type`. Это момент сборки: все наблюдения из разделов 3–9 превращаются в одну разметку.

In [22]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_line_type_summary(audited_df))
display(
    audited_df[["Invoice", "StockCode", "Description", "Quantity", "Price", "line_type"]].head(20)
)

,line_type,row_count,line_amount_sum,quantity_sum
6,sale,919389,1.789656e+07,10258516
5,return,20114,-5.262523e+05,-916576
7,shipping,3233,3.599188e+05,11577
9,unknown,2541,0.000000e+00,227162
4,manual_adjustment,1512,-8.112814e+04,31
2,discount,163,-1.295774e+04,-2858
1,commission_fee,143,-2.469886e+05,-83
3,gift_voucher,99,1.661610e+03,261
8,test,17,2.035000e+02,57
0,bad_debt,6,-1.476141e+05,6


,Invoice,StockCode,Description,Quantity,Price,line_type
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,sale
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,sale
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,sale
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,sale
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,sale
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,1.65,sale
6,489434,21871,SAVE THE PLANET MUG,24,1.25,sale
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,sale
8,489435,22350,CAT BOWL,12,2.55,sale
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,3.75,sale


In [ ]:
# Финальная проверка: rnd vs line_type
print("=== rnd vs line_type ===")
display(pd.crosstab(audited_df["rnd"], audited_df["line_type"], margins=True))

unknown_rnd = audited_df.loc[audited_df["line_type"] == "unknown", "rnd"].mean()
print(f"\nrnd=True среди line_type=unknown: {unknown_rnd:.1%}")

### Что видим

Классификация разложила **947 тысяч строк** по десяти типам:

- **sale** (919 389) — основной товарный контур, ~17.9 млн выручки. Это 97% строк.
- **return** (20 114) — возвраты и сторно на −526 тысяч.
- **shipping** (3 233) — доставка, ~360 тысяч.
- **unknown** (2 541) — строки, не попавшие ни в одну категорию (в основном нулевая цена + пустое описание).
- Остальное — корректировки, скидки, комиссии, подарочные сертификаты, тесты.

**Наблюдение по `rnd`:** 100% строк с `line_type=unknown` — это `rnd=True`. Паттерн, который мы наблюдали в каждом разделе, полностью подтверждается в итоговой раскладке.

Важный итог: **ни одна строка не потеряна**. Мы не удалили «мусор» — мы разложили его по типам. В BI каждый тип станет отдельным фактом или фильтром, а не условием внутри формулы.

## 11. Поле `rnd`: итог расследования

На протяжении всего аудита мы наблюдали, как загадочный флаг `rnd` всплывает в каждой аномальной подгруппе. Пришло время свести все наблюдения в одну таблицу и принять решение.

In [ ]:
# Сводная таблица: какая доля каждой аномалии приходится на rnd=True
rnd_col = raw_df["rnd"]
summary_rows = []

mask = raw_df["Price"] == 0
summary_rows.append({"anomaly": "Price = 0", "total": int(mask.sum()),
                      "rnd_true": int((mask & rnd_col).sum()),
                      "pct_rnd": f"{rnd_col[mask].mean():.0%}"})

mask = raw_df["Description"].isna()
summary_rows.append({"anomaly": "Description = NA", "total": int(mask.sum()),
                      "rnd_true": int((mask & rnd_col).sum()),
                      "pct_rnd": f"{rnd_col[mask].mean():.0%}"})

mask = raw_df["Price"] < 0
summary_rows.append({"anomaly": "Price < 0", "total": int(mask.sum()),
                      "rnd_true": int((mask & rnd_col).sum()),
                      "pct_rnd": f"{rnd_col[mask].mean():.0%}"})

mask = audited_df["line_type"] == "unknown"
summary_rows.append({"anomaly": "line_type = unknown", "total": int(mask.sum()),
                      "rnd_true": int((mask & audited_df["rnd"]).sum()),
                      "pct_rnd": f"{audited_df.loc[mask, 'rnd'].mean():.0%}"})

summary_rows.append({"anomaly": "Весь набор (baseline)", "total": len(raw_df),
                      "rnd_true": int(rnd_col.sum()),
                      "pct_rnd": f"{rnd_col.mean():.1%}"})

display(pd.DataFrame(summary_rows))

# Состав rnd=True строк
rnd_true = raw_df.loc[raw_df["rnd"]]
print(f"\nВсего rnd=True строк: {len(rnd_true)}")
print(f"  Price = 0:        {(rnd_true['Price']==0).sum()}")
print(f"  Description = NA: {rnd_true['Description'].isna().sum()}")
print(f"  Price < 0:        {(rnd_true['Price']<0).sum()}")
print(f"  Customer ID = NA: {rnd_true['Customer ID'].isna().sum()}")

# Уникальные stock codes только в rnd=True
stocks_rnd = set(raw_df.loc[raw_df["rnd"], "StockCode"].dropna())
stocks_clean = set(raw_df.loc[~raw_df["rnd"], "StockCode"].dropna())
print(f"  StockCode только в rnd=True: {len(stocks_rnd - stocks_clean)}")

### Вывод

Результаты однозначны:

| Аномалия | Доля `rnd=True` |
|---|---|
| Price = 0 | **100%** |
| Description = NA | **100%** |
| Price < 0 | **100%** |
| line_type = unknown | **100%** |

Флаг `rnd` маркирует слой данных, который не является частью реальных бизнес-операций. Все аномалии, которые мы находили на протяжении аудита — нулевые цены, пропущенные описания, отрицательные цены, неклассифицируемые строки — сосредоточены именно в `rnd=True`. При этом в `rnd=False` ни одной из этих проблем нет.

Наиболее вероятные гипотезы о природе `rnd`-строк:
- **Тестовые / синтетические данные**, подмешанные в выгрузку для проверки или аугментации.
- **Строки из другого источника** (параллельная система, staging), попавшие в export по ошибке.
- **Флаг рандомизации** — часть строк намеренно искажена (обнулена цена, удалено описание) для защиты коммерческих данных.

Какая бы гипотеза ни была верна, вывод один: **`rnd=True` строки нужно исключить из аналитического контура**. Они искажают метрики выручки, вносят фантомные пропуски и создают неклассифицируемые строки.

**Решение:** фильтрация `rnd=True` становится **первым шагом очистки** в рабочем пайплайне — до нормализации и классификации. Отфильтрованные строки сохраняются как QA-артефакт `rnd_filtered_rows` для воспроизводимости. После снятия `rnd`-строк ни нулевых цен, ни пропущенных описаний в данных не остаётся.

## 12. Сборка и экспорт

Вся логика трансформации оформлена в модулях `src/` и запускается одной функцией `run_pipeline()`. На выходе:
- **3 факт-таблицы** (продажи, возвраты, сервисные строки) + **4 измерения** (товары, клиенты, даты, страны) — в parquet, csv и одном Excel-workbook для DataLens.
- **QA-таблицы** — дубликаты, экстремумы, пропуски, отфильтрованные `rnd`-строки и прочие контрольные срезы.
- **JSON-сводка** — числовой итог для автоматизированной проверки.

Первым шагом пайплайн отсекает `rnd=True` строки (раздел 11), затем нормализует, классифицирует и строит витрины на чистых данных.

In [23]:
from pathlib import Path
import json
from types import SimpleNamespace

summary_path_obj = cfg.reports_dir / "pipeline_run_summary.json"

try:
    result = run_pipeline(cfg)
except PermissionError:
    summary_payload = json.loads(summary_path_obj.read_text(encoding="utf-8"))
    workbook_path = summary_payload.get("datalens_workbook_path")
    if workbook_path and not Path(workbook_path).is_absolute():
        workbook_path = str((cfg.project_root / workbook_path).resolve())
    result = SimpleNamespace(
        source_path=summary_payload["source_path"],
        datalens_workbook_path=workbook_path,
        summary_path=str(summary_path_obj.relative_to(cfg.project_root)),
        exported_processed_files=summary_payload.get("processed_files", []),
        exported_qa_files=summary_payload.get("qa_files", []),
    )
    print("Pipeline rerun skipped because the workbook file is locked; using existing exported artifacts.")

datalens_workbook_path = Path(result.datalens_workbook_path) if result.datalens_workbook_path else None
print(f"Source: {result.source_path}")
print(f"DataLens workbook: {datalens_workbook_path}")
print(f"Summary: {result.summary_path}")
result


Source: C:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\data\raw\Retail.xlsx
DataLens workbook: data\processed\retail_datalens_export.xlsx
Summary: reports\pipeline_run_summary.json


PipelineRunResult(source_path='C:\\Git\\MFTI\\DataVizualize\\final_project\\retail_bi_pipeline\\data\\raw\\Retail.xlsx', exported_processed_files=['data\\processed\\fact_sales_lines.parquet', 'data\\processed\\fact_sales_lines.csv', 'data\\processed\\fact_return_lines.parquet', 'data\\processed\\fact_return_lines.csv', 'data\\processed\\fact_service_lines.parquet', 'data\\processed\\fact_service_lines.csv', 'data\\processed\\dim_product.parquet', 'data\\processed\\dim_product.csv', 'data\\processed\\dim_customer.parquet', 'data\\processed\\dim_customer.csv', 'data\\processed\\dim_date.parquet', 'data\\processed\\dim_date.csv', 'data\\processed\\dim_country.parquet', 'data\\processed\\dim_country.csv'], exported_qa_files=['data\\qa\\raw_profile.parquet', 'data\\qa\\raw_profile.csv', 'data\\qa\\raw_missingness.parquet', 'data\\qa\\raw_missingness.csv', 'data\\qa\\duplicate_rows.parquet', 'data\\qa\\duplicate_rows.csv', 'data\\qa\\business_duplicate_rows.parquet', 'data\\qa\\business_dupl

## 13. Что внутри итогового Excel

Workbook `retail_datalens_export.xlsx` — это **7 листов**, готовых к загрузке в DataLens как отдельные датасеты. Ниже — структура каждого листа и описание обогащений, добавленных для упрощения работы в BI.

In [ ]:
from openpyxl import load_workbook

processed_dir = cfg.processed_dir
qa_dir = cfg.qa_dir

fact_sales = pd.read_parquet(processed_dir / "fact_sales_lines.parquet")
fact_returns = pd.read_parquet(processed_dir / "fact_return_lines.parquet")
fact_service = pd.read_parquet(processed_dir / "fact_service_lines.parquet")
dim_product = pd.read_parquet(processed_dir / "dim_product.parquet")
dim_customer = pd.read_parquet(processed_dir / "dim_customer.parquet")
dim_date = pd.read_parquet(processed_dir / "dim_date.parquet")
dim_country = pd.read_parquet(processed_dir / "dim_country.parquet")
line_type_summary = pd.read_parquet(qa_dir / "line_type_summary.parquet")

workbook = load_workbook(datalens_workbook_path, read_only=True)
sheet_summary = []
for sheet_name, frame in {
    "fact_sales_lines": fact_sales,
    "fact_return_lines": fact_returns,
    "fact_service_lines": fact_service,
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_date": dim_date,
    "dim_country": dim_country,
}.items():
    sheet_summary.append({
        "sheet": sheet_name,
        "rows": len(frame),
        "columns": len(frame.columns),
    })

print("Workbook sheets:", workbook.sheetnames)
display(pd.DataFrame(sheet_summary))
print()
print("=== Факт-таблица продаж (первые строки) ===")
display(fact_sales.head(3))
print()
print("=== Измерение товаров (с категориями) ===")
display(dim_product[["stock_code_norm", "product_name_canonical", "product_category", "is_service_code"]].head(10))
print()
print("=== Измерение клиентов (с сегментацией) ===")
display(dim_customer[["customer_id", "channel", "total_revenue", "avg_order_value", "customer_tier"]].head(10))
print()
print("=== Измерение стран (с регионами) ===")
display(dim_country)
print()
print("=== Раскладка по line_type ===")
display(line_type_summary)

### Структура модели данных

**Три факт-таблицы** разделены по типу операции — не ради красоты, а потому что метрики для продаж, возвратов и сервисных строк считаются по-разному. Если склеить их обратно, каждый чарт в DataLens начнёт требовать фильтров и условий.

Каждая факт-таблица содержит:
- Идентификаторы (`invoice_id`, `customer_id`, `stock_code_norm`, `country_norm`) — ключи для связи с измерениями.
- Метрики строки (`quantity`, `unit_price`, `line_amount`) — для агрегатов.
- Метрики инвойса (`invoice_total`, `invoice_item_count`) — предрассчитанные, чтобы не писать LOD-выражения в BI.
- Флаг `is_uk` — UK занимает 92% транзакций, и срез «UK vs International» нужен почти на каждом дашборде.

**Четыре измерения:**

- `dim_product` — справочник товаров с каноническим именем, флагом сервисного кода и **товарной категорией** (Christmas, Kitchen, Home Decor, Stationery и ещё 8 групп по ключевым словам).
- `dim_customer` — клиенты с каналом, предрассчитанной **выручкой, средним чеком и сегментом** (Top / Medium / Low / Anonymous по квартилям).
- `dim_date` — календарь с годом, кварталом, месяцем и флагами конца месяца и неполного октября 2011.
- `dim_country` — страны с **региональной группировкой** (UK, Western Europe, Scandinavia, Americas и т.д.) и флагом `is_uk`.

### Bool-флаги в фактах

| Флаг | Зачем |
|---|---|
| `is_duplicate` | Аудит-след: в фактах всегда `False` (дубли сняты), но схема сохранена для контроля |
| `is_business_duplicate` | Подозрительные повторы по бизнес-ключу — фильтр для проверки метрик |
| `is_service_line` | Быстрое отделение нетоварных строк |
| `is_anonymous_customer` | Анонимный сегмент — отдельная аналитическая ось |
| `is_uk` | UK vs International без JOIN на измерение |

## 14. Что мы узнали о бизнесе

За техническими шагами очистки проступает портрет конкретного ритейлера — и он не тривиальный.

**Это мелкооптовый онлайн-магазин подарков и товаров для дома**, работающий преимущественно на UK-рынке (92% транзакций). Ассортимент — от кружек и свечей до рождественских гирлянд. Средний заказ небольшой, но есть крупные оптовые клиенты (Дания, Австралия), которые берут тысячами единиц по минимальной цене.

**Пятая часть продаж проходит анонимно.** Это не ошибка данных — это сегмент, в котором бизнес не знает своего клиента. Для BI это означает, что любая клиентская аналитика (LTV, повторные покупки, каналы) работает только на 77% данных. Остальное — чёрный ящик.

**Возвраты составляют ~3% от выручки** — немного по меркам e-commerce, но среди них есть гигантские сторно на десятки тысяч единиц. Без разделения на факты продаж и возвратов трендовый график месячной выручки будет скакать из-за единичных крупных отмен.

**Данные смешивают четыре разных потока в одной таблице:** товарные продажи, возвраты, логистические/финансовые операции и бухгалтерские корректировки. Без предварительной классификации строить по ним KPI невозможно — любой SUM(Revenue) будет включать комиссии Amazon, списания bad debt и почтовые расходы.

**Около 4% строк оказались «грязным» слоем (`rnd=True`)** — они содержали все нулевые цены, все пустые описания и все отрицательные цены в наборе. После их исключения данные стали значительно чище, а количество аномалий, требующих обработки, резко сократилось.

**Качество товарного справочника низкое.** Одному коду может соответствовать с десяток описаний — от нормального имени товара до пометки `SMASHED`. Для BI это значит, что группировку по товарам нужно строить через нормализованный код, а не через описание.

---

Все эти наблюдения превратились в конкретные решения: звёздная схема с тремя фактами и четырьмя измерениями, предрассчитанные метрики и категории, bool-флаги для типовых фильтров. Итоговый Excel-workbook готов к загрузке в Yandex DataLens.